# Presentación · Trayectorias laborales tras la formación

Preguntas del proyecto respondidas **en vivo** sobre **1.506.434 experiencias** de **284.247 personas** (`data/03_processed/empleos_limpio.csv` — JobHop v2 + ESCO).
Cada pregunta se responde con una gráfica. Reproducir: *Kernel → Restart & Run All*.


In [ ]:
from pathlib import Path
import sys

import numpy as np  # noqa: F401
import pandas as pd

try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except Exception:
    import matplotlib
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(*objetos):
        for o in objetos:
            print(o.to_string() if hasattr(o, "to_string") else o)

PROYECTO = Path.cwd()
while not (PROYECTO / "data").exists() and PROYECTO != PROYECTO.parent:
    PROYECTO = PROYECTO.parent
sys.path.insert(0, str(PROYECTO / "src" / "filtro"))
import filtros as F  # noqa: E402
import indicadores as I  # noqa: E402

RUTA = PROYECTO / "data" / "03_processed" / "empleos_limpio.csv"
empleos = pd.read_csv(RUTA, dtype=str, encoding="utf-8", keep_default_na=False, na_values=[""])
for _flag in ["es_unknown_ocupacion", "es_rescatado", "es_vigente"]:
    empleos[_flag] = empleos[_flag].eq("True")

filas, columnas = empleos.shape
personas = int(empleos["resume_id"].nunique())

# Línea base contra la auditoría del proyecto: sin esto el libro no es interpretable.
assert filas == 1_506_434 and columnas == 14 and personas == 284_247
assert int(empleos["es_unknown_ocupacion"].sum()) == 104_993
assert int(empleos["es_rescatado"].sum()) == 10_176
assert int(empleos["es_vigente"].sum()) == 76_180

ESTILO_BARRA = dict(edgecolor="white", linewidth=0.8)
def _malla(ax):
    ax.grid(axis="y", alpha=0.3)

print(f"Línea base OK: {filas:,} filas · {columnas} columnas · {personas:,} personas")


## 1 · Línea base — ¿qué datos responden las preguntas?


In [ ]:
base_nivel = empleos["university_level"].value_counts()
base_banderas = empleos[["es_unknown_ocupacion", "es_rescatado", "es_vigente"]].sum()

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
base_nivel.plot(kind="bar", ax=axs[0], rot=15, **ESTILO_BARRA)
axs[0].set_title(f"Experiencias por nivel educativo (N = {filas:,})")
axs[0].set_ylabel("experiencias")
base_banderas.index = ["unknown", "rescatado", "vigente"]
base_banderas.plot(kind="bar", ax=axs[1], rot=0, **ESTILO_BARRA)
axs[1].set_title("Banderas del pipeline")
axs[1].set_ylabel("experiencias")
for ax in axs:
    _malla(ax)
plt.tight_layout()
plt.show()


## 2 · Preguntas de exploración


### P1 · ¿Cuántas personas con doctorado y cuántas experiencias hay?


In [ ]:
phd = empleos[empleos["university_level"].eq("PhD")]
conteo = pd.Series({"personas": phd["resume_id"].nunique(), "experiencias": len(phd)})
ax = conteo.plot(kind="bar", rot=0, **ESTILO_BARRA)
ax.set_title(f"PhD · {phd['resume_id'].nunique():,} personas · {len(phd):,} experiencias")
ax.set_ylabel("")
_malla(ax)
plt.tight_layout()
plt.show()


### P2 · ¿En qué grupo ocupacional (ISCO) se concentran más experiencias?


In [ ]:
top_g = I.top_valores(empleos["isco_group_label"], 10)
fig, ax = plt.subplots(figsize=(9, 3.8))
top_g["n"].plot(kind="barh", ax=ax, **ESTILO_BARRA)
ax.set_title(f"Top 10 grupos ISCO · el más frecuente concentra {top_g['%_de_con_etiqueta'].iloc[0]:.1f} %")
ax.set_xlabel("experiencias")
plt.tight_layout()
plt.show()


### P3 · ¿Cuántas experiencias inician en 2015–2019 y qué % sigue vigente?


In [ ]:
recientes = F.filtrar_por_periodo(empleos, 2015, 2019)
por_anio = recientes["start_date"].str.split(" ").str[1].astype(int).value_counts().sort_index()
ax = por_anio.plot(kind="bar", rot=0, **ESTILO_BARRA)
ax.set_title(f"Inicios 2015–2019 ({len(recientes):,} experiencias) · {100 * recientes['es_vigente'].mean():.1f} % vigentes")
ax.set_ylabel("experiencias")
_malla(ax)
plt.tight_layout()
plt.show()


### P4 · ¿Cuántas experiencias están vigentes (fin = Present)?


In [ ]:
vigentes = int(empleos["es_vigente"].sum())
conteo = pd.Series({"vigentes (fin = Present)": vigentes, "finalizadas": filas - vigentes})
ax = conteo.plot(kind="bar", rot=0, color=["#4C72B0", "#C0C0C0"], **ESTILO_BARRA)
ax.set_title(f"Vigentes = {100 * vigentes / filas:.1f} % (su duración está censurada, queda fuera del análisis de duración)")
ax.set_ylabel("experiencias")
_malla(ax)
plt.tight_layout()
plt.show()


### P5 · ¿Cuánto pesa “sin clasificar” y en qué niveles es mayor?


In [ ]:
sin_area = F.filtrar_sin_area_isco(empleos)
pct_nivel = (
    sin_area["university_level"].value_counts()
    / empleos["university_level"].value_counts() * 100
).reindex(["Secondary school", "Bachelor", "Master", "PhD", "No reportado"]).round(1)
ax = pct_nivel.plot(kind="bar", rot=15, **ESTILO_BARRA, color="#C44E52")
ax.set_title(f"Experiencias sin área ISCO por nivel (global {100 * len(sin_area) / filas:.2f} %)")
ax.set_ylabel("% del nivel")
_malla(ax)
plt.tight_layout()
plt.show()


## 3 · Relaciones


### R1 · ¿El nivel educativo cambia el repertorio de grupos ISCO?


In [ ]:
X = I.crosstab_nivel_grupo(empleos, top=8)
fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(X, cmap="Blues", aspect="auto")
ax.set_xticks(range(X.shape[1]))
ax.set_xticklabels(X.columns, rotation=30, ha="right", fontsize=7)
ax.set_yticks(range(X.shape[0]))
ax.set_yticklabels(X.index, fontsize=8)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        ax.text(j, i, f"{X.iloc[i, j]:.0f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="% de experiencias del nivel")
ax.set_title("Grupos ISCO dominantes por nivel educativo (% del nivel)")
plt.tight_layout()
plt.show()


### R2 · ¿Cambia la duración según el grupo ISCO o el nivel educativo?


In [ ]:
por_grupo = I.duracion_mediana_por_grupo(empleos, top=8)["mediana"].sort_values()
por_nivel = I.duracion_mediana_por_nivel(empleos)["mediana"]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
por_grupo.plot(kind="barh", ax=axs[0], **ESTILO_BARRA)
axs[0].set_title("Mediana de duración por grupo ISCO (top)")
axs[0].set_xlabel("trimestres")
por_nivel.plot(kind="bar", ax=axs[1], rot=15, **ESTILO_BARRA)
axs[1].set_title("Mediana de duración por nivel")
axs[1].set_ylabel("trimestres")
for ax in axs:
    _malla(ax)
plt.tight_layout()
plt.show()


### R3 · ¿Qué tan lineal es la trayectoria (conservan el grupo ISCO)?


In [ ]:
mismo_global = I.proporcion_mismo_grupo(empleos)
mismo_nivel = pd.Series({
    nivel: I.proporcion_mismo_grupo(empleos[empleos["university_level"].eq(nivel)])
    for nivel in ["Secondary school", "Bachelor", "Master", "PhD", "No reportado"]
})
ax = mismo_nivel.plot(kind="bar", rot=15, color="#C44E52", **ESTILO_BARRA)
ax.axhline(mismo_global, color="black", ls="--", lw=1)
ax.set_title(f"Transiciones que conservan el grupo ISCO (global {mismo_global:.1f} % — línea punteada)")
ax.set_ylabel("% de transiciones")
_malla(ax)
plt.tight_layout()
plt.show()


### R4 · ¿Hay periodos sin empleo registrado (proxy de desempleo)? ¿Varían por nivel?


In [ ]:
_, resumen_b = I.brechas_entre_empleos(empleos)
por_nivel_b = I.brechas_por_nivel(empleos)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
por_nivel_b["%_con_brecha"].plot(kind="bar", ax=axs[0], rot=15, **ESTILO_BARRA)
axs[0].set_title(f"Personas con ≥ 1 brecha (global {100 * resumen_b['personas_con_brecha'] / resumen_b['personas']:.1f} %)")
axs[0].set_ylabel("% de personas")
por_nivel_b["mediana_brecha"].plot(kind="bar", ax=axs[1], rot=15, **ESTILO_BARRA)
axs[1].set_title(f"Mediana de la mayor brecha (global {resumen_b['mediana_trimestres']:.0f} trimestres)")
axs[1].set_ylabel("trimestres")
for ax in axs:
    _malla(ax)
plt.tight_layout()
plt.show()


## 4 · Mini investigación


### Q1 · ¿La concentración ocupacional cambia con la educación?


In [ ]:
concentracion = pd.Series({
    nivel: round(
        100 * empleos.loc[empleos["university_level"].eq(nivel), "isco_group_label"]
        .dropna().value_counts(normalize=True).head(3).sum(), 1)
    for nivel in ["Secondary school", "Bachelor", "Master", "PhD", "No reportado"]
})
ax = concentracion.plot(kind="bar", rot=15, color="#55A868", **ESTILO_BARRA)
ax.set_title("Concentración: % de experiencias del nivel cubiertas por sus top-3 grupos ISCO")
ax.set_ylabel("%")
_malla(ax)
plt.tight_layout()
plt.show()


### Q2 · ¿Cuánto dura una experiencia? ¿Cambia con el nivel?


In [ ]:
con_dur = empleos.assign(dur=I.duracion_trimestres(empleos)).dropna(subset=["dur"])
con_dur = con_dur.assign(
    nivel=pd.Categorical(
        con_dur["university_level"],
        categories=["Secondary school", "Bachelor", "Master", "PhD", "No reportado"],
        ordered=True,
    )
)
fig, ax = plt.subplots(figsize=(9, 3.8))
con_dur.boxplot(column="dur", by="nivel", ax=ax, showfliers=False)
ax.set_ylim(0, 30)
ax.set_title(f"Duración de experiencias finalizadas por nivel (mediana global {con_dur['dur'].median():.0f} trimestres, IQR 2–11)")
ax.set_ylabel("trimestres")
plt.suptitle("")
plt.tight_layout()
plt.show()


### Q3 · ¿Dónde aterriza la 1.ª experiencia → 2.ª?


In [ ]:
prim = I.primera_a_segunda(empleos).head(8)
pares = prim.set_index(
    prim["isco_group_label_primera"] + "  →  " + prim["isco_group_label_segunda"]
)["n"].sort_values()
ax = pares.plot(kind="barh", figsize=(9, 3.8), **ESTILO_BARRA)
ax.set_title(f"Pares 1.ª→2.ª experiencia más frecuentes (N = {int(prim['n'].sum()):,})")
ax.set_xlabel("personas")
plt.tight_layout()
plt.show()


### Q4 · ¿Cómo se distribuyen los periodos sin empleo (brechas)?


In [ ]:
brechas, _ = I.brechas_entre_empleos(empleos)
positivas = brechas.loc[brechas["con_brecha"], "brecha"]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(positivas.clip(upper=40), bins=range(0, 43), color="#4C72B0", edgecolor="white", alpha=0.85)
ax.set_xticks(range(0, 45, 5))
ax.set_title(
    f"Distribución de brechas · {resumen_b['brechas']:,} brechas · mediana {resumen_b['mediana_trimestres']:.0f} trimestres · "
    f"máx {resumen_b['max_trimestres']:.0f}"
)
ax.set_ylabel("brechas")
_malla(ax)
plt.tight_layout()
plt.show()


## 5 · En una línea
· La educación ordena el repertorio: cintos a los que accede cada nivel (R1, Q1).
· Pocas experiencias duran mucho: mediana 5 trimestres, pero hay colas largas (Q2).
· Las trayectorias son conservadoras: ~18 % de las transiciones conservan el grupo ISCO (R3).
· Casi 60 % de las personas tiene al menos un periodo sin empleo registrado (R4, Q4).

**Limitaciones**: duración censurada (fin = Present), ocupación previa con dato faltante → los indicadores de transición/brecha se calculan solo sobre pares completamente etiquetados.
